# Chapter 48 — Top-K Retrieval: Controlling How Many Documents Are Returned

Learning objectives
- Understand what K means in Top-K retrieval.
- Compare semantic search with different values of K.
- Build a reusable, validated Top-K retriever.
- Understand the trade-off between retrieving too few and too many passages.
This notebook is self-contained: it creates its own documents, embeddings, ChromaDB client, and collection. Run cells from top to bottom.

Top-1: The single closest matching document.



Top-3: The three closest matching documents.



Top-5: The five closest matching documents.

In [ ]:
%pip install -q chromadb sentence-transformers numpy


# Step 1: Import Libraries

In [8]:
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer

/Users/vimal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# Step 2 : Create Sample documents and metadata

In [10]:
documents = [
    "Banks monitor cross-border payments to identify suspicious transaction patterns.",
    "Money laundering conceals the illegal origin of funds.",
    "Customer due diligence verifies customer identity and assesses customer risk.",
    "Credit risk models estimate the probability that a borrower will default.",
    "Machine learning models detect unusual patterns in financial transactions.",
    "Fraud detection systems identify payments that may be unauthorized or deceptive.",
    "Investigators review unusual international wire transfers for signs of financial crime.",
    "Know Your Customer checks help banks understand who their customers are.",
    "Sanctions screening checks whether a person or organization appears on a sanctions list.",
    "A neural network can classify images and process natural language.",
    "Transaction monitoring rules flag payments that match defined risk scenarios.",
    "Rapid movement of funds between multiple accounts may require investigation."
]

ids = [f"doc_{i:02d}" for i in range(1, len(documents) + 1)]
metadatas = [
    {"category": "AML", "source": "monitoring"},
    {"category": "AML", "source": "financial_crime"},
    {"category": "KYC", "source": "customer_risk"},
    {"category": "Credit Risk", "source": "risk_model"},
    {"category": "AML", "source": "machine_learning"},
    {"category": "Fraud", "source": "fraud_detection"},
    {"category": "AML", "source": "investigation"},
    {"category": "KYC", "source": "customer_risk"},
    {"category": "Sanctions", "source": "screening"},
    {"category": "AI", "source": "general_ai"},
    {"category": "AML", "source": "monitoring"},
    {"category": "AML", "source": "investigation"}
]

print("Documents:", len(documents))
print("IDs:", len(ids))
print("Metadata records:", len(metadatas))

Documents: 12
IDs: 12
Metadata records: 12


In [11]:
assert len(documents) == len(ids) == len(metadatas)

# Step 3 : Create Embeddings

In [12]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(documents)

In [13]:
embeddings.shape

(12, 384)

In [14]:
embeddings.shape[1]

384

# Step 4 : Store Documents in ChromaDB

In [15]:
client = chromadb.EphemeralClient()
collection = client.create_collection(
    name="chapter_48_top_k_retrieval",
    metadata={"hnsw:space": "cosine"}
)
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings.tolist()
)
print("Documents stored:", collection.count())

Documents stored: 12


# Step 5 : Define the query and encode it 

In [16]:
query = "How can a bank detect suspicious international payments?"
query_embedding = model.encode(query)

# Step 6 : Top -1 retrieval 

In [17]:
result_top_1 = collection.query(query_embeddings=query_embedding, n_results=1)

In [18]:
result_top_1

{'ids': [['doc_01']],
 'embeddings': None,
 'documents': [['Banks monitor cross-border payments to identify suspicious transaction patterns.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'category': 'AML', 'source': 'monitoring'}]],
 'distances': [[0.235812246799469]]}

In [21]:
for rank, document in enumerate(result_top_1["documents"][0], start=1):
    index = rank - 1
    print(f"Rank: {rank}")
    print("Document:", document)
    print("Metadata:", result_top_1["metadatas"][0][index])
    print(f"Cosine distance: {result_top_1['distances'][0][index]:.4f}")


Rank: 1
Document: Banks monitor cross-border payments to identify suspicious transaction patterns.
Metadata: {'category': 'AML', 'source': 'monitoring'}
Cosine distance: 0.2358


# Step 7 :Compare Top-1, Top-3, Top-5

In [28]:
def show_top_k(query_text, top_k):
    query_vector = model.encode(query_text)
    results = collection.query(query_embeddings=query_vector,
                               n_results = min(top_k, collection.count()))

    print(f"Query: {query_text}")
    print(f"Requested K: {top_k}")
    print(f"Returned: {len(results['documents'][0])}")

    for rank, document in enumerate(results["documents"][0], start=1):
        i = rank - 1
        print(f"{rank}. {document}")
        print(f"   Category: {results['metadatas'][0][i]['category']}")
        print(f"   Distance: {results['distances'][0][i]:.4f}")
    print()

for k in [1, 3, 5, 10]:
    show_top_k(query, k)

Query: How can a bank detect suspicious international payments?
Requested K: 1
Returned: 1
1. Banks monitor cross-border payments to identify suspicious transaction patterns.
   Category: AML
   Distance: 0.2358

Query: How can a bank detect suspicious international payments?
Requested K: 3
Returned: 3
1. Banks monitor cross-border payments to identify suspicious transaction patterns.
   Category: AML
   Distance: 0.2358
2. Investigators review unusual international wire transfers for signs of financial crime.
   Category: AML
   Distance: 0.3411
3. Fraud detection systems identify payments that may be unauthorized or deceptive.
   Category: Fraud
   Distance: 0.4247

Query: How can a bank detect suspicious international payments?
Requested K: 5
Returned: 5
1. Banks monitor cross-border payments to identify suspicious transaction patterns.
   Category: AML
   Distance: 0.2358
2. Investigators review unusual international wire transfers for signs of financial crime.
   Category: AML
   